# 00 Build overview

| Field | Value |
|---|---|
| **Purpose** | Confirm which build every notebook uses and show the raw file before any cleaning |
| **Assignment requirement** | Project foundation: reproducibility and evidence lineage |
| **Inputs** | `run_manifest.json`, `raw_column_profile.csv`, `raw_daily_coverage.csv` |
| **Outputs** | None; a verification notebook |
| **Build identity** | Printed by the first code cell |

**Rule for every notebook:** the first code cell calls `start()`. It stops with an error if the raw data, the settings, the code, the pinned environment definition or any output file differ from the recorded build. A running environment that differs from `environment.yml` gives a warning. Every table a notebook reads through `project.read_csv()` has its hash checked again at read time.

In [1]:
from job_market.notebook import start

project = start()

RUN_ID: 20261004T130021Z-5894edd8d2
Source: data/raw/all_upwork_jobs_2024-02-07-2024-03-24.csv (244,828 rows)
Taxonomy version: 1.0
Verified outputs: 46 file(s), all hashes match
Environment: matches environment.yml


## Build identity

The manifest records four fingerprints - the source file, the settings, the code and the pinned environment file `environment.yml` - plus the package versions used for this build.

In [2]:
import pandas as pd

manifest = project.manifest
identity = pd.Series({
    "run_id": manifest["run_id"],
    "source file": manifest["source"]["path"],
    "source SHA-256": manifest["fingerprints"]["source_sha256"],
    "config SHA-256": manifest["fingerprints"]["config_sha256"],
    "code SHA-256": manifest["fingerprints"]["code_sha256"],
    "environment SHA-256": manifest["fingerprints"]["environment_sha256"],
    "python": manifest["environment"]["python"],
    **{f"package: {k}": v for k, v in manifest["environment"]["packages"].items()},
})
identity.to_frame("value")

,value
run_id,20261004T130021Z-5894edd8d2
source file,data/raw/all_upwork_jobs_2024-02-07-2024-03-24...
source SHA-256,6513e9a744f15726bdb40d6435a60834e864ace08a75e1...
config SHA-256,9d7b83bb238a078221ec867a3558e6da42a3f01b5b89e6...
code SHA-256,fbef31546a0a6ff34f6677e2fcc7cacec0e853ed34f24e...
environment SHA-256,16395497efe48498ffe44d037ec2020318fdbc12dca3fd...
python,3.11.16
package: flask,3.0.3
package: ipykernel,6.29.5
package: matplotlib,3.9.2


## Raw file and cleaning totals

The ingest step changes no values. It checks the schema and records the shape of the raw data. The cleaning step adds the totals below; each cleaning decision has its own documented reason in Notebook 01.

Only the foundation steps (ingest and clean) are shown here. The job categories and their accuracy belong to Notebook 03, where they are explained and tested.

In [3]:
pd.Series(project.step_metrics("ingest", "clean")).to_frame("value")

,value
calendar_days_with_postings,89
duplicate_links_non_missing,0
exact_duplicate_rows,0
first_posting_utc,2023-11-02T09:22:02Z
fixed_price_rows,103891
hourly_rows,140937
last_posting_utc,2024-03-24T14:16:47Z
missing_link,1
missing_title,1
negative_numeric_values,0


In [4]:
project.read_csv("raw_column_profile.csv")

,column,dtype,non_missing,missing,missing_pct,distinct_values
0,title,object,244827,1,0.0004,205470
1,link,object,244827,1,0.0004,244827
2,published_date,object,244828,0,0.0000,233640
3,is_hourly,bool,244828,0,0.0000,2
4,hourly_low,float64,102422,142406,58.1657,114
5,hourly_high,float64,98775,146053,59.6553,227
6,budget,float64,103891,140937,57.5657,768
7,country,object,239751,5077,2.0737,212


## Daily coverage

The first and last timestamp of each UTC day show how complete that day's collection is. A normal day spans close to 24 hours; a day where collection starts late or stops early spans far less. The cleaning stage uses this table to decide which days a time-series analysis can trust.

In [5]:
coverage = project.read_csv("raw_daily_coverage.csv")
print(f"Days with at least one posting: {len(coverage)}")
coverage.tail(10)

Days with at least one posting: 89


,date,postings,first_posting_utc,last_posting_utc,observed_span_hours
79,2024-03-15,6329,2024-03-15T00:00:26Z,2024-03-15T23:59:07Z,23.98
80,2024-03-16,4605,2024-03-16T00:00:14Z,2024-03-16T23:58:33Z,23.97
81,2024-03-17,4478,2024-03-17T00:00:00Z,2024-03-17T23:59:55Z,24.00
82,2024-03-18,6811,2024-03-18T00:00:07Z,2024-03-18T23:59:55Z,24.00
83,2024-03-19,6906,2024-03-19T00:00:03Z,2024-03-19T23:59:08Z,23.98
84,2024-03-20,6839,2024-03-20T00:00:02Z,2024-03-20T23:59:27Z,23.99
85,2024-03-21,6814,2024-03-21T00:00:00Z,2024-03-21T23:57:21Z,23.96
86,2024-03-22,6208,2024-03-22T00:03:29Z,2024-03-22T23:59:21Z,23.93
87,2024-03-23,4531,2024-03-23T00:00:08Z,2024-03-23T23:59:55Z,24.00
88,2024-03-24,2185,2024-03-24T00:01:33Z,2024-03-24T14:16:47Z,14.25
